<a href="https://colab.research.google.com/github/peterbabulik/QuantumWalker/blob/main/QCA_1_bit_adder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install qiskit qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.0/109.0 kB 6.7 MB/s eta 0:00:00


In [2]:

import numpy as np
import qiskit
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
from qiskit.circuit.library import CXGate, CZGate, RZZGate, RXXGate, RYYGate, SwapGate, HGate, XGate, IGate, SGate, TGate, CCXGate # Toffoli
import matplotlib.pyplot as plt
import time
import os
from typing import List, Optional, Tuple, Dict
from tqdm import tqdm

# --- Parameters ---
TYPE_A = 0 # Represents bit 0 or a default/neutral state
TYPE_B = 1 # Represents bit 1 or an active/specific state

# --- Standard Gate Objects (Instructions) ---
GATES = {
    "H": HGate(), "X": XGate(), "I": IGate(), "S": SGate(), "T": TGate(),
    "CX": CXGate(), "CZ": CZGate(), "SWAP": SwapGate(), "CCX": CCXGate(),
    "RZZ_PIO4": RZZGate(np.pi/4), "RZZ_PIO2": RZZGate(np.pi/2),
    "RXX_PIO4": RXXGate(np.pi/4), "RYY_PIO4": RYYGate(np.pi/4),
}

class QuantumCellularAutomaton:
    def __init__(self, num_qunodes: int, initial_types: np.ndarray,
                 interaction_config: Dict,
                 type_update_rule_config: Dict,
                 initial_quantum_state_prep: str = "all_zero" # Default to |0>
                ):
        self.num_qunodes = num_qunodes
        if len(initial_types) != num_qunodes: raise ValueError("initial_types length mismatch.")
        self.current_types = initial_types.copy()
        self.type_history = [self.current_types.copy()]
        self.quantum_outcome_history = []
        self.interaction_config = interaction_config
        self.type_update_rule_config = type_update_rule_config
        self.initial_quantum_state_prep = initial_quantum_state_prep
        self.simulator = AerSimulator(method='statevector')
        self.circuit_visualization_list = []

    def _get_gate_object(self, gate_name_or_obj):
        if isinstance(gate_name_or_obj, str):
            return GATES.get(gate_name_or_obj.upper(), GATES["I"])
        elif isinstance(gate_name_or_obj, qiskit.circuit.Instruction):
            return gate_name_or_obj
        return GATES["I"]

    def _build_interaction_circuit(self) -> QuantumCircuit:
        qc_step = QuantumCircuit(self.num_qunodes, name=f"QCA_Interaction_Step")

        if self.initial_quantum_state_prep == "hadamard_all":
            for i in range(self.num_qunodes): qc_step.h(i)
        elif self.initial_quantum_state_prep == "basis_from_type":
            for i in range(self.num_qunodes):
                if self.current_types[i] == TYPE_B: qc_step.x(i)
        elif self.initial_quantum_state_prep == "mixed_A0_BH":
             for i in range(self.num_qunodes):
                if self.current_types[i] == TYPE_B: qc_step.h(i)

        if self.num_qunodes > 0: qc_step.barrier(label="init_prep")

        interaction_applied_this_step = False
        processed_qubits_in_specific_interactions = set()

        # The interaction_config can be a list of operations to apply in sequence
        # or a dictionary for more complex lookup (like type-based).
        # For adder circuit, we assume it's a list of (gate_spec, qubit_indices_tuple)
        # or a dictionary where keys are qubit_indices_tuples.

        # If interaction_config is a list (sequence of operations)
        if isinstance(self.interaction_config.get("circuit_sequence"), list):
            for op_spec in self.interaction_config["circuit_sequence"]:
                if len(op_spec) == 2:
                    gate_spec, qubit_indices = op_spec
                    op_to_apply = self._get_gate_object(gate_spec)
                    if op_to_apply.name.lower() != 'id':
                        try:
                            qc_step.append(op_to_apply, list(qubit_indices))
                            interaction_applied_this_step = True
                        except qiskit.exceptions.QiskitError as e:
                            print(f"Warning: Could not apply gate {op_to_apply.name} from sequence. Error: {e}")
                else:
                    print(f"Warning: Invalid operation spec in circuit_sequence: {op_spec}")

        else: # Assume dictionary-based config (original logic)
            for key, gate_spec in self.interaction_config.items():
                if key == "apply_general_nearest_neighbor" or key == "circuit_sequence": continue # Skip config flags

                if isinstance(key, tuple) and all(isinstance(idx, int) for idx in key):
                    qubit_indices = list(key)
                    if not all(0 <= idx < self.num_qunodes for idx in qubit_indices): continue
                    if len(set(qubit_indices)) != len(qubit_indices): continue

                    op_to_apply = self._get_gate_object(gate_spec)
                    if op_to_apply.name.lower() != 'id':
                        try:
                            qc_step.append(op_to_apply, qubit_indices)
                            interaction_applied_this_step = True
                            for idx in qubit_indices: processed_qubits_in_specific_interactions.add(idx)
                        except qiskit.exceptions.QiskitError as e:
                            print(f"Warning: Could not apply gate {op_to_apply.name} to {qubit_indices}. Error: {e}")

            if self.interaction_config.get("apply_general_nearest_neighbor", False):
                for i in range(self.num_qunodes):
                    q_j_idx = (i + 1) % self.num_qunodes
                    already_processed_i = i in processed_qubits_in_specific_interactions
                    already_processed_j = q_j_idx in processed_qubits_in_specific_interactions

                    if self.num_qunodes >= 2 and not (already_processed_i and already_processed_j):
                        type1, type2 = self.current_types[i], self.current_types[q_j_idx]
                        type_key_str = "".join(map(str, sorted((type1, type2))))
                        gate_spec_for_type = self.interaction_config.get(type_key_str, GATES["I"])
                        op_to_apply_type_based = self._get_gate_object(gate_spec_for_type)
                        if op_to_apply_type_based.name.lower() != 'id':
                            try: qc_step.append(op_to_apply_type_based, [i, q_j_idx]); interaction_applied_this_step = True
                            except qiskit.exceptions.QiskitError as e: print(f"Warning: Type-based gate {op_to_apply_type_based.name} err: {e}")

        if interaction_applied_this_step and self.num_qunodes > 0 :
            qc_step.barrier(label="interactions")
        return qc_step

    def _apply_type_update_rule(self, quantum_outcomes_p1: np.ndarray) -> np.ndarray:
        new_types = self.current_types.copy()
        rule_params = self.type_update_rule_config
        rule_name = rule_params.get("name", "direct_p1_to_type")

        if rule_name == "direct_p1_to_type":
            threshold = rule_params.get("p1_threshold_for_type_B", 0.5)
            for i in range(self.num_qunodes):
                new_types[i] = TYPE_B if quantum_outcomes_p1[i] > threshold else TYPE_A
        elif rule_name == "simple_threshold": # Your existing one
            # ... (implementation from previous code) ...
            for i in range(self.num_qunodes):
                my_p1 = quantum_outcomes_p1[i]
                if self.current_types[i] == TYPE_A and my_p1 > rule_params.get("thresh_A_to_B", 0.55):
                    new_types[i] = TYPE_B
                elif self.current_types[i] == TYPE_B and my_p1 < rule_params.get("thresh_B_to_A", 0.45):
                    new_types[i] = TYPE_A
        elif rule_name == "neighbor_influence_threshold": # From your previous advanced version
            # ... (implementation from previous code) ...
            for i in range(self.num_qunodes):
                my_p1 = quantum_outcomes_p1[i]
                left_idx = (i - 1 + self.num_qunodes) % self.num_qunodes
                right_idx = (i + 1) % self.num_qunodes
                left_p1, right_p1 = quantum_outcomes_p1[left_idx], quantum_outcomes_p1[right_idx]
                left_type, right_type = self.current_types[left_idx], self.current_types[right_idx]

                if self.current_types[i] == TYPE_A:
                    flip_to_B = False
                    if my_p1 > rule_params.get("thresh_A_to_B_self", 0.6): flip_to_B = True
                    elif (left_type == TYPE_B and left_p1 < rule_params.get("thresh_B_neighbor_influence_on_A", 0.4)) or \
                         (right_type == TYPE_B and right_p1 < rule_params.get("thresh_B_neighbor_influence_on_A", 0.4)):
                        if np.random.rand() < rule_params.get("prob_A_flip_by_B_neighbor", 0.1): flip_to_B = True
                    if flip_to_B: new_types[i] = TYPE_B
                elif self.current_types[i] == TYPE_B:
                    flip_to_A = False
                    if my_p1 < rule_params.get("thresh_B_to_A_self", 0.4): flip_to_A = True
                    elif (left_type == TYPE_A and left_p1 > rule_params.get("thresh_A_neighbor_influence_on_B", 0.6)) or \
                         (right_type == TYPE_A and right_p1 > rule_params.get("thresh_A_neighbor_influence_on_B", 0.6)):
                        if np.random.rand() < rule_params.get("prob_B_flip_by_A_neighbor", 0.1): flip_to_A = True
                    if flip_to_A: new_types[i] = TYPE_A
        return new_types

    def step(self) -> np.ndarray:
        interaction_qc = self._build_interaction_circuit()
        if not self.circuit_visualization_list:
            self.circuit_visualization_list.append(interaction_qc.copy())

        quantum_outcomes_p1 = np.zeros(self.num_qunodes)
        meaningful_ops = False
        if interaction_qc.num_qubits > 0:
            for instruction in interaction_qc.data:
                op_name = instruction.operation.name.lower()
                # A more robust check for "meaningful": not identity or barrier AND not just initial prep if it's the only thing.
                # This logic can be tricky. For now, assume if there are ops beyond barrier/id, it's meaningful.
                if op_name not in ['barrier', 'id', 'snapshot', 'save_statevector']:
                    meaningful_ops = True
                    break

        if interaction_qc.num_qubits > 0:
            # Always simulate if there are meaningful ops or if it's the very first step (to get initial P1s from prep)
            if meaningful_ops or not self.quantum_outcome_history:
                final_statevector_obj = Statevector(interaction_qc)
                for i in range(self.num_qunodes):
                    probs_qi = final_statevector_obj.probabilities([i])
                    quantum_outcomes_p1[i] = probs_qi[1] if len(probs_qi) == 2 else 0.0
            else: # No new meaningful ops, P1s are effectively from the initial prep of this "empty" step
                temp_qc_for_p1 = QuantumCircuit(self.num_qunodes)
                if self.initial_quantum_state_prep == "hadamard_all": [temp_qc_for_p1.h(i) for i in range(self.num_qunodes)]
                elif self.initial_quantum_state_prep == "basis_from_type":
                    for i in range(self.num_qunodes):
                        if self.current_types[i] == TYPE_B: temp_qc_for_p1.x(i)
                elif self.initial_quantum_state_prep == "mixed_A0_BH":
                    for i in range(self.num_qunodes):
                        if self.current_types[i] == TYPE_B: temp_qc_for_p1.h(i)

                if temp_qc_for_p1.num_qubits > 0:
                    sv = Statevector(temp_qc_for_p1)
                    for i in range(self.num_qunodes):
                        quantum_outcomes_p1[i] = (sv.probabilities([i])[1] if len(sv.probabilities([i]))==2 else 0.0)

        self.quantum_outcome_history.append(quantum_outcomes_p1.copy())
        new_types = self._apply_type_update_rule(quantum_outcomes_p1)
        self.current_types = new_types
        self.type_history.append(self.current_types.copy())
        return quantum_outcomes_p1

    def run(self, num_steps: int, verbose: bool = False):
        # ... (implementation from previous code, no changes needed here) ...
        if verbose:
            print(f"Starting QCA simulation: {self.num_qunodes} qunodes, {num_steps} steps.")
            print(f"Initial Types: {self.current_types}")
            print(f"Initial Quantum State Prep: {self.initial_quantum_state_prep}")
            print(f"Interaction Config: {self.interaction_config}")
            print(f"Type Update Rule Config: {self.type_update_rule_config}")

        iterator = range(num_steps)
        if not verbose:
            try: iterator = tqdm(range(num_steps), desc=f"QCA Steps ({self.num_qunodes}Q)")
            except ImportError: print("tqdm not found. Progress bar disabled.")

        for step_num in iterator:
            p1_outcomes = self.step()
            if verbose:
                print(f"Step {step_num + 1}/{num_steps} -> P(|1⟩): {np.round(p1_outcomes, 3)}, New Types: {self.current_types}")
            elif not isinstance(iterator, range) and isinstance(iterator,tqdm):
                pass
            elif (step_num + 1) % (max(1, num_steps // 10)) == 0 or step_num == num_steps - 1:
                print(f"  QCA Step {step_num + 1}/{num_steps} completed...")

        if isinstance(iterator, tqdm): iterator.close()
        if verbose or not ('tqdm' in str(type(iterator))):
             print("\n--- QCA Simulation Complete ---")


    def get_type_history_as_array(self) -> np.ndarray:
        return np.array(self.type_history)

    def plot_type_evolution(self, title_suffix: str = "", experiment_label: str = ""):
        # ... (implementation from previous code, no changes needed here) ...
        history_array = self.get_type_history_as_array()
        if history_array.size == 0:
            print("No type history to plot.")
            return

        fig_width = max(10, history_array.shape[0] * 0.2)
        fig_height = max(6, self.num_qunodes * 0.6)

        plt.figure(figsize=(fig_width, fig_height))
        plt.imshow(history_array.T, cmap='viridis', aspect='auto', interpolation='nearest', origin='lower')
        plt.xlabel(f"Time Step (Depth = {history_array.shape[0] - 1})")
        plt.ylabel("Qunode Index")
        title = f"Evolution of QCA Types"
        if experiment_label: title += f" - {experiment_label}"
        if title_suffix: title += title_suffix
        plt.title(title, fontsize=14)
        plt.colorbar(label="Type (0=A, 1=B)", ticks=[TYPE_A, TYPE_B])
        if self.num_qunodes > 0: plt.yticks(np.arange(self.num_qunodes))
        plt.tight_layout()
        plt.show()


    def visualize_example_interaction_circuit(self):
        # ... (implementation from previous code, no changes needed here) ...
        if self.circuit_visualization_list:
            print("\n--- Example Interaction Circuit (dynamics from first step) ---")
            circuit_to_draw = self.circuit_visualization_list[0]
            if circuit_to_draw.num_qubits > 0:
                try:
                    from IPython.display import display
                    print("Attempting to display circuit with Matplotlib drawer...")
                    display(circuit_to_draw.draw(output='mpl', fold=-1, initial_state=True))
                except ImportError:
                    print("IPython display not available. Using text drawer.")
                    print(circuit_to_draw.draw(output='text', fold=-1))
                except Exception as e:
                    print(f"Error drawing circuit with mpl: {e}. Falling back to text.")
                    print(circuit_to_draw.draw(output='text', fold=-1))
            else:
                print("Interaction circuit has no qubits to draw.")
        else:
            print("No interaction circuit was stored (e.g., simulation not run or empty).")


# --- Main execution for Adder Test ---
if __name__ == "__main__":
    output_base_dir = "qca_full_adder_test_output"
    if not os.path.exists(output_base_dir): os.makedirs(output_base_dir)

    # Quantum Full Adder Circuit Logic
    # Qunode mapping:
    # Q0: Input A
    # Q1: Input B
    # Q2: Input Carry_in (Cin)
    # Q3: Output Sum (S) - starts |0>
    # Q4: Output Carry_out (Cout) - starts |0>
    # This circuit computes S on Q3 and Cout on Q4 without modifying Q0, Q1, Q2.
    # It's a common textbook implementation.
    full_adder_interaction_logic = {
        "circuit_sequence": [
            # Carry part first onto Q4
            ("CCX", (0, 1, 4)), # Q4 = A AND B (if Q4 was |0>)
            ("CX", (0, 1)),     # Q1 = A XOR B
            ("CCX", (1, 2, 4)), # Q4 = Q4 XOR ( (A XOR B) AND Cin )
                                # So, Q4 = (A AND B) XOR ( (A XOR B) AND Cin ) -> this is Carry_out
            # Sum part onto Q3
            ("CX", (1, 2)),     # Q2 = (A XOR B) XOR Cin -> This is Sum, but on Q2. We need to undo Q1 change.
                                # Let's correct the sum part to target Q3 directly and be independent.
            # Revert Q1 for sum calculation
            ("CX", (0,1)),      # Q1 back to B (A^B^A = B)

            # Sum = A XOR B XOR Cin, calculated on Q3
            ("CX", (0, 3)),     # Q3 = A (since Q3 was |0>)
            ("CX", (1, 3)),     # Q3 = A XOR B
            ("CX", (2, 3)),     # Q3 = A XOR B XOR Cin --> This is Sum on Q3
        ],
        "apply_general_nearest_neighbor": False # Explicit circuit only
    }
    # The above carry logic for Q4 is simplified.
    # A standard quantum full adder (e.g., Cuccaro et al.) is more complex for ripple-carry.
    # Let's use a known minimal one that is correct for single bit.
    # MAJ (a,b,c) = c XOR ((a XOR c) AND (b XOR c))
    # SUM (a,b,c) = a XOR b XOR c

    # Corrected Full Adder logic (targets Q3 for Sum, Q4 for Carry)
    # Using a standard ripple-carry adder design:
    # q0=Cin, q1=A, q2=B, q3=Sum, q4=temporary carry (ancilla-like, could be final carry for 1 bit)
    # For our mapping: A=q0, B=q1, Cin=q2, Sum=q3, Cout=q4
    # This means we need to be careful about which qubit is which.
    # Let's use the version from Nielsen & Chuang (Fig 4.9) adapted.
    # Theirs: Cin, A, B, Sum (B is modified to Sum), Ancilla (gets Cout)
    # Our desired: A, B, Cin -> Sum, Cout (inputs preserved)
    # This requires 2 Toffolis and 2 CNOTs for the carry, and 3 CNOTs for sum.
    # Or simpler: Sum = A^B^Cin ; Cout = MAJ(A,B,Cin)
    # MAJ(x,y,z) can be done with CCX(x,y,anc), CCX(x,z,anc), CCX(y,z,anc) if anc is |0>
    # and then a final operation to get it to Cout. Or simpler:
    # MAJ(x,y,z) = x XOR y XOR z XOR SUM(x,y,z) --- No, this is not MAJ.
    # MAJ(x,y,z) = (x AND y) XOR (x AND z) XOR (y AND z)

    # Let's use a known circuit structure for SUM and CARRY_OUT
    # q0=A, q1=B, q2=Cin, q3=SUM, q4=CARRY_OUT
    # Both q3 and q4 must start in |0> state.
    full_adder_circuit_logic_textbook = {
        "circuit_sequence": [
            # Calculate Carry-Out on q4
            ("CCX", (0, 1, 4)),  # q4 = A & B (if q4 was |0>)
            ("CX", (0, 1)),      # q1_new = A ^ B
            ("CCX", (2, 1, 4)),  # q4 = q4 ^ (Cin & (A^B)) --- Corrects q4 to be Carry_Out
            ("CX", (0, 1)),      # q1 back to B (A^B^A = B) --- Restore q1

            # Calculate Sum on q3 (A^B^Cin)
            ("CX", (0, 3)),      # q3 = A
            ("CX", (1, 3)),      # q3 = A ^ B
            ("CX", (2, 3)),      # q3 = A ^ B ^ Cin
        ],
         "apply_general_nearest_neighbor": False
    }


    print("\n--- Testing 1-bit FULL Adder QCA (Sum and Carry computation) ---")
    print("Qunodes: 0=A, 1=B, 2=Cin, 3=Sum_Target, 4=Carry_Target")

    all_tests_passed_count = 0
    total_tests = 0

    for a_val_in in [0, 1]:
        for b_val_in in [0, 1]:
            for ci_val_in in [0, 1]:
                total_tests += 1
                num_qunodes_adder = 5
                initial_types_adder = np.zeros(num_qunodes_adder, dtype=int)
                initial_types_adder[0] = TYPE_B if a_val_in == 1 else TYPE_A
                initial_types_adder[1] = TYPE_B if b_val_in == 1 else TYPE_A
                initial_types_adder[2] = TYPE_B if ci_val_in == 1 else TYPE_A

                qca_instance = QuantumCellularAutomaton(
                    num_qunodes=num_qunodes_adder,
                    initial_types=initial_types_adder,
                    interaction_config=full_adder_circuit_logic_textbook, # Using the full adder logic
                    type_update_rule_config={"name": "direct_p1_to_type", "p1_threshold_for_type_B": 0.5},
                    initial_quantum_state_prep="basis_from_type"
                )

                print(f"\nInput: A={a_val_in}, B={b_val_in}, Cin={ci_val_in}")
                print(f"Initial QCA Types: {qca_instance.current_types}")

                qca_instance.run(num_steps=1, verbose=False)

                final_types_from_qca = qca_instance.current_types
                qca_sum_bit_output = final_types_from_qca[3]
                qca_carry_bit_output = final_types_from_qca[4]

                classical_sum_expected = (a_val_in ^ b_val_in) ^ ci_val_in
                classical_carry_expected = (a_val_in & b_val_in) | (ci_val_in & (a_val_in ^ b_val_in))

                print(f"QCA Final Types:   {final_types_from_qca}")
                print(f"QCA Output -> Sum (Q3): {qca_sum_bit_output}, Carry_out (Q4): {qca_carry_bit_output}")
                print(f"Classical Expected -> Sum: {classical_sum_expected}, Carry_out: {classical_carry_expected}")

                sum_correct = (qca_sum_bit_output == classical_sum_expected)
                carry_correct = (qca_carry_bit_output == classical_carry_expected)

                if sum_correct and carry_correct:
                    print("Result: CORRECT")
                    all_tests_passed_count +=1
                else:
                    print("Result: INCORRECT")
                    if not sum_correct: print("  -> Sum bit mismatch.")
                    if not carry_correct: print("  -> Carry bit mismatch.")
                print("-" * 30)

                if a_val_in == 1 and b_val_in == 1 and ci_val_in == 0 and total_tests==7: # Visualize one case
                    qca_instance.visualize_example_interaction_circuit()


    print("\n--- Summary for Full Adder Test ---")
    if all_tests_passed_count == total_tests:
        print(f"All {total_tests} full adder tests passed successfully!")
    else:
        print(f"{all_tests_passed_count} out of {total_tests} full adder tests passed.")

    print("\nThis test used a pre-defined quantum circuit for a full adder, executed in a single QCA 'step'.")


--- Testing 1-bit FULL Adder QCA (Sum and Carry computation) ---
Qunodes: 0=A, 1=B, 2=Cin, 3=Sum_Target, 4=Carry_Target

Input: A=0, B=0, Cin=0
Initial QCA Types: [0 0 0 0 0]


QCA Steps (5Q): 100%|██████████| 1/1 [00:00<00:00, 45.16it/s]


QCA Final Types:   [0 0 0 0 0]
QCA Output -> Sum (Q3): 0, Carry_out (Q4): 0
Classical Expected -> Sum: 0, Carry_out: 0
Result: CORRECT
------------------------------

Input: A=0, B=0, Cin=1
Initial QCA Types: [0 0 1 0 0]


QCA Steps (5Q): 100%|██████████| 1/1 [00:00<00:00, 410.32it/s]


QCA Final Types:   [0 0 1 1 0]
QCA Output -> Sum (Q3): 1, Carry_out (Q4): 0
Classical Expected -> Sum: 1, Carry_out: 0
Result: CORRECT
------------------------------

Input: A=0, B=1, Cin=0
Initial QCA Types: [0 1 0 0 0]


...

QCA Steps (5Q): 100%|██████████| 1/1 [00:00<00:00, 91.17it/s]

QCA Final Types:   [1 1 0 0 1]
QCA Output -> Sum (Q3): 0, Carry_out (Q4): 1
Classical Expected -> Sum: 0, Carry_out: 1
Result: CORRECT
------------------------------

--- Example Interaction Circuit (dynamics from first step) ---
Attempting to display circuit with Matplotlib drawer...


IPython display not available. Using text drawer.
     ┌───┐ init_prep                                     interactions 
q_0: ┤ X ├─────░───────■────■─────────■────■──────────────────░───────
     ├───┤     ░       │  ┌─┴─┐     ┌─┴─┐  │                  ░       
q_1: ┤ X ├─────░───────■──┤ X ├──■──┤ X ├──┼────■─────────────░───────
     └───┘     ░       │  └───┘  │  └───┘  │    │             ░       
q_2: ──────────░───────┼─────────■─────────┼────┼────■────────░───────
               ░       │         │       ┌─┴─┐┌─┴─┐┌─┴─┐      ░       
q_3: ──────────░───────┼─────────┼───────┤ X ├┤ X ├┤ X ├──────░───────
               ░     ┌─┴─┐     ┌─┴─┐     └───┘└───┘└───┘      ░       
q_4: ──────────░─────┤ X ├─────┤ X ├──────────────────────────░───────
               ░     └───┘     └───┘                          ░       

Input: A=1, B=1, Cin=1
Initial QCA Types: [1 1 1 0 0]


QCA Steps (5Q): 100%|██████████| 1/1 [00:00<00:00, 259.63it/s]

QCA Final Types:   [1 1 1 1 1]
QCA Output -> Sum (Q3): 1, Carry_out (Q4): 1
Classical Expected -> Sum: 1, Carry_out: 1
Result: CORRECT
------------------------------

--- Summary for Full Adder Test ---
All 8 full adder tests passed successfully!

This test used a pre-defined quantum circuit for a full adder, executed in a single QCA 'step'.
